# MichiganCast Demo 03: Multimodal Training and Manual Tuning

This notebook demonstrates modular multimodal training, manual hyperparameter exploration, and experiment level tracking.

## 0. Objectives and Conclusions

**Objectives**
1. Review the manual experiment matrix in `configs/experiments`.
2. Launch multimodal training through the CLI entry point.
3. Analyze epoch level metrics and experiment artifacts.

**Conclusion Template**
- Which hyperparameter pattern produced the strongest validation behavior.
- Which settings improved recall without destabilizing precision.
- Which run should be promoted for export.

## 1. Inputs and Outputs

| Type | Path |
|---|---|
| Experiment configs | `configs/experiments/t21_exp*.yaml` |
| Training entry | `src/models/multimodal/train.py` |
| Epoch metrics | `artifacts/reports/multimodal_epoch_metrics.csv` |
| Metric figure | `artifacts/reports/multimodal_train_validation_metrics.png` |
| Training summary | `artifacts/reports/multimodal_train_summary.json` |
| Experiment runs | `artifacts/experiments/<run_id>/metrics.json` |

In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import yaml

EXP_DIR = Path('configs/experiments')
REPORT_DIR = Path('artifacts/reports')
RUN_DIR = Path('artifacts/experiments')

print('experiment config dir exists:', EXP_DIR.exists())
print('report dir exists:', REPORT_DIR.exists())
print('experiment run dir exists:', RUN_DIR.exists())

## 2. Method and Implementation Using src Modules

Execution is disabled by default. Set `DEMO_EXECUTE=True` when you intentionally run training from the notebook.

In [ ]:
DEMO_EXECUTE = False

def run_or_print(cmd: str):
    print('\n$ ' + cmd)
    if not DEMO_EXECUTE:
        print('[skip] DEMO_EXECUTE=False, only showing command.')
        return None
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
    return proc.returncode

### 2.1 Review Manual Hyperparameter Matrix T21

**Purpose** Compare manually designed experiments across horizon, lookback, model width, regularization, optimizer settings, and decision threshold.

In [ ]:
exp_files = sorted(EXP_DIR.glob('t21_exp*.yaml'))
print('experiment config count:', len(exp_files))

rows = []
for p in exp_files:
    cfg = yaml.safe_load(p.read_text(encoding='utf-8'))
    rows.append({
        'experiment_id': cfg.get('metadata', {}).get('experiment_id', p.stem),
        'horizon_hours': cfg.get('task', {}).get('horizon_hours'),
        'meteo_lookback': cfg.get('task', {}).get('meteo_lookback_steps'),
        'image_lookback': cfg.get('task', {}).get('image_lookback_steps'),
        'conv_hidden_dim': cfg.get('model', {}).get('conv_hidden_dim'),
        'meteo_hidden_dim': cfg.get('model', {}).get('meteo_hidden_dim'),
        'dropout': cfg.get('model', {}).get('dropout'),
        'learning_rate': cfg.get('train', {}).get('learning_rate'),
        'loss_type': cfg.get('train', {}).get('loss', {}).get('type'),
        'decision_threshold': cfg.get('train', {}).get('decision_threshold'),
    })

df_matrix = pd.DataFrame(rows)
display(df_matrix)

### 2.2 Launch Multimodal Training T20 and T22

**Purpose** Run one tracked experiment through the modular training pipeline and persist run metadata.

In [ ]:
cmd_train = (
    'scripts/run_in_pytorch_env.sh python -m src.models.multimodal.train '
    '--input-csv data/interim/traverse_city_daytime_clean_v1.csv '
    '--image-dir data/processed/images/lake_michigan_64_png '
    '--output-dir artifacts/reports '
    '--checkpoint-path models/pytorch/michigancast_multimodal_best.pth '
    '--device auto '
    '--apple-metal-opt '
    '--experiment-name demo_tuning '
    '--experiment-root artifacts/experiments'
)
run_or_print(cmd_train)

### 2.3 Inspect Experiment Tracking T22

**Purpose** Confirm each run has an auditable record of config, metrics, and checkpoint references.

In [ ]:
run_dirs = sorted([p for p in RUN_DIR.glob('*') if p.is_dir()])
print('run count:', len(run_dirs))
for p in run_dirs[-5:]:
    print('-', p.name)

if run_dirs:
    latest = run_dirs[-1]
    metrics_path = latest / 'metrics.json'
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        print('\nlatest run id:', metrics.get('run_id'))
        print('best_val_loss:', metrics.get('best_val_loss'))
        print('test_pr_auc:', metrics.get('test_metrics', {}).get('pr_auc'))
    else:
        print('latest run has no metrics.json')

## 3. Results and Interpretation

Use epoch curves and summary metrics to evaluate:
1. Convergence speed.
2. Overfitting risk.
3. Precision recall tradeoff at the selected threshold.

In [ ]:
epoch_csv = REPORT_DIR / 'multimodal_epoch_metrics.csv'
summary_json = REPORT_DIR / 'multimodal_train_summary.json'
curve_png = REPORT_DIR / 'multimodal_train_validation_metrics.png'

print('epoch csv exists:', epoch_csv.exists())
print('summary exists:', summary_json.exists())
print('curve png exists:', curve_png.exists())

if epoch_csv.exists():
    df_epoch = pd.read_csv(epoch_csv)
    display(df_epoch.tail(10))
else:
    print('No epoch metrics CSV yet. Run section 2.2 first.')

In [ ]:
if summary_json.exists():
    summary = json.loads(summary_json.read_text(encoding='utf-8'))
    print('device:', summary.get('runtime_profile', {}).get('device'))
    print('batch_size:', summary.get('runtime_profile', {}).get('batch_size'))
    print('epochs_ran:', summary.get('fit_result', {}).get('epochs_ran'))
    print('best_val_loss:', summary.get('fit_result', {}).get('best_val_loss'))
    print('test_pr_auc:', summary.get('test_metrics', {}).get('pr_auc'))
else:
    print('No training summary yet. Run section 2.2 first.')

### 3.1 Presentation Highlights

1. Explain why manual tuning choices are hypothesis driven.
2. Show how tracking artifacts improve reproducibility.
3. Connect final model selection to business risk metrics.

## 4. Engineering Notes and Next Steps

- Add structured experiment comparison reports for top k runs.
- Extend reproducibility with pinned dataset version references.
- Add optional mixed precision benchmarks for hardware profiles.

## Appendix. Reproducible Commands

```bash
scripts/run_in_pytorch_env.sh python -m src.models.multimodal.train --device auto --apple-metal-opt --experiment-name demo_tuning
scripts/run_in_pytorch_env.sh python -m src.models.multimodal.train --device auto --apple-metal-opt --experiment-name demo_tuning_alt --threshold 0.45 --learning-rate 0.0008
```